# PROMPT ENGINEERING AND RAG
## SESSION 4

Assignment questions and answers


## Question 1

AI-AUGMENTED LEARNING

STEP 1 — BUILD WITH AI

Write a Python program that reads a plain-text food-delivery
refund-policy file from disk, splits it into fixed-size chunks,
generates sentence-transformer embeddings, and stores them in a FAISS
index.

Requirements: - Accept a user question typed at the command line,
retrieve the top two most relevant chunks, and construct a structured
RAG prompt using those chunks as context. - Print the retrieved chunks,
assembled prompt, and a simulated placeholder answer. A hardcoded answer
is acceptable; no live API call is required. - Run in a loop so the user
can ask multiple questions without restarting the script, and exit
cleanly when the user types ‘quit’.

STEP 2 — TEST & DEBUG (WITHOUT AI)

Test the code independently, identify at least one bug, limitation, or
improvement in the AI-generated solution, and fix it yourself. Consider
chunks splitting mid-sentence, FAISS returning fewer results than k when
the index has fewer chunks, missing instructions to answer only from
supplied context, or failure to strip whitespace before checking for
‘quit’.

SUBMISSION REQUIREMENTS: - The exact prompt(s) given to the AI tool. -
The AI’s original code and corrected version. - A 3–4 line note
explaining what you changed and why the AI’s version needed it.

**Answer:**

### Prompt Given to the AI Tool
"Write a Python script that reads a plain-text refund policy file, splits it into fixed-size character chunks, generates sentence-transformer embeddings, and stores them in a FAISS index. Run a loop accepting user queries until they type 'quit', retrieve the top 2 chunks via FAISS, print the chunks, print an assembled RAG prompt, and print a simulated response."

---

### AI's Original Code

```python
import faiss
from sentence_transformers import SentenceTransformer

# Load file and chunk
with open("refund_policy.txt", "r") as f:
    text = f.read()

chunk_size = 200
chunks = [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]

# Embed and index
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Query loop
while True:
    query = input("Ask a question: ")
    if query == "quit":
        break

    q_embed = model.encode([query])
    D, I = index.search(q_embed, k=2)

    retrieved = [chunks[idx] for idx in I[0]]
    print("\nRetrieved Chunks:\n", retrieved)

    prompt = f"Context: {' '.join(retrieved)}\nQuestion: {query}\nAnswer:"
    print("\nPrompt:\n", prompt)
    print("\nAnswer: [Simulated] Refunds are processed in 5-7 business days.\n")
```

---

### Corrected Version

```python
import os
import faiss
from sentence_transformers import SentenceTransformer

if not os.path.exists("refund_policy.txt"):
    with open("refund_policy.txt", "w") as f:
        f.write("Refunds are allowed within 14 days of delivery. Late items qualify for a full refund.")

with open("refund_policy.txt", "r") as f:
    text = f.read()

# Fix 1: Split by sentence to avoid breaking context mid-sentence
raw_sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
chunks = [s + "." for s in raw_sentences] or ["No policy content found."]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

while True:
    query = input("Ask a question (or 'quit'): ")
    # Fix 2: Handle whitespace and case-sensitivity
    if query.strip().lower() == "quit":
        break
    if not query.strip():
        continue

    # Fix 3: Bound k to available chunks to prevent FAISS index error
    k = min(2, len(chunks))
    q_embed = model.encode([query])
    D, I = index.search(q_embed, k=k)

    retrieved = [chunks[idx] for idx in I[0] if idx != -1]
    print("\nRetrieved Chunks:", retrieved)

    # Fix 4: Explicit system instructions added to the prompt
    prompt = (
        "Answer the question using ONLY the context below. If unknown, say 'Not found'.\n"
        f"Context: {' '.join(retrieved)}\n"
        f"Question: {query.strip()}\n"
        "Answer:"
    )
    print("\nPrompt:\n", prompt)
    print("Simulated Answer: [System would generate answer based on retrieved context]\n")
```

---

### Changes and Explanation
The AI's original code sliced strings by character count, causing broken mid-sentence fragments, and used a raw string equality check (`query == "quit"`) that failed on trailing spaces or capitalization. It also set `k=2` statically without verifying if the document had at least two chunks, which causes invalid index lookups on tiny files. Finally, the constructed prompt lacked strict instructions to answer strictly using the provided context, which risks hallucinations in actual RAG implementations.